ENSEMBLE LEARNING

Question 1: What is Ensemble Learning in machine learning? Explain the key idea behind it.

Ans - Ensemble Learning is a machine-learning approach that combines predictions from multiple models (called base learners) to produce a final prediction. Instead of relying on one model, it uses the strengths of several models to improve generalization, stability, and predictive performance.

The key idea is that a group of models can often make better predictions than a single model, especially when their errors are not identical. For classification, predictions may be combined by majority voting or averaged probabilities. For regression, predictions are commonly averaged. Bagging, Random Forest, and Boosting are widely used ensemble methods.

Advantages include reduced variance, improved accuracy, and greater robustness. However, ensembles can require more computation and may be less interpretable than a single model.

Question 2: What is the difference between Bagging and Boosting?

Ans - Bagging (Bootstrap Aggregating) trains multiple models independently and often in parallel. Each model receives a different bootstrap sample of the training data. Their predictions are combined by voting or averaging. Bagging primarily reduces variance and helps stabilize high-variance learners. Random Forest is a bagging-based method that also selects random subsets of features when splitting trees.

Boosting trains models sequentially. Each new model focuses on patterns or errors that previous models handled poorly. Depending on the algorithm, this is achieved by reweighting observations or fitting residuals/gradients. The combined model is a weighted sum or vote of the learners. Boosting can reduce bias and produce strong predictive performance, but may overfit if not regularized or tuned.



Question 3: What is bootstrap sampling and what role does it play in Bagging methods like Random Forest?

Ans - Bootstrap sampling creates a new sample by randomly drawing observations from the original training dataset with replacement. The bootstrap sample usually has the same number of rows as the original dataset, but some observations may appear more than once while others are left out.

In Bagging, a different bootstrap sample is used to train each base learner. This makes the learners differ from one another. Their predictions are then aggregated, which can reduce variance compared with relying on one model.

In a Random Forest, each tree is trained on a bootstrap sample (by default in common implementations), and a random subset of features is considered at each split. The combination of row sampling and feature randomness decorrelates trees and helps the forest generalize. Observations omitted from a tree's bootstrap sample can also be used for out-of-bag evaluation.

Question 4: What are Out-of-Bag (OOB) samples and how is OOB score used to evaluate ensemble models?

Ans - Out-of-Bag (OOB) samples are training observations that are not selected in the bootstrap sample used to train a particular tree. For a sample, predictions can be collected from the trees for which that sample was OOB, and those predictions can be aggregated.

The OOB score compares these aggregated predictions with the true labels or values. For classification, it is commonly an accuracy-like score; for regression, an appropriate prediction score can be used. It provides an internal estimate of generalization performance without requiring a separate validation set, which can make it useful during model development.

OOB estimates are not a replacement for a final untouched test set, especially when many modeling decisions are made using the OOB score. In scikit-learn, OOB evaluation can be enabled with options such as `oob_score=True` when supported by the estimator and sampling configuration.

Question 5: Compare feature importance analysis in a single Decision Tree vs. a Random Forest.

Ans - A single Decision Tree computes feature importance from the total reduction in the splitting criterion (such as Gini impurity) attributable to each feature across the tree. It is relatively easy to inspect and explain, but its importance values can be unstable: small changes in training data may produce a different tree and different rankings.

A Random Forest aggregates importance contributions across many trees, so its rankings are generally more stable than those from one tree. It can capture patterns across multiple samples and feature subsets. In scikit-learn, `feature_importances_` usually reports mean decrease in impurity aggregated across the forest.

Both methods can favor variables with many possible split points or high cardinality. Correlated features can share or obscure importance, and impurity-based importance is not necessarily causal. Permutation importance on held-out data can provide a useful complementary check.

In [9]:
#Question 6: Write a Python program to
#load the Breast Cancer dataset using sklearn.datasets.load_breast_cancer(),
#train a Random Forest Classifier, and
#print the top 5 most important features based on feature importance scores.

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2,
    random_state=42, stratify=data.target
)
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

ranked = sorted(
    zip(data.feature_names, model.feature_importances_),
    key=lambda item: item[1], reverse=True
)
print("Top 5 important features:")
for name, score in ranked[:5]:
    print(f"{name}: {score:.4f}")

Top 5 important features:
worst perimeter: 0.1331
worst area: 0.1281
worst concave points: 0.1081
mean concave points: 0.0944
worst radius: 0.0906


In [10]:
#Question 7: Write a Python program to
#train a Bagging Classifier using Decision Trees on the Iris dataset,
#evaluate its accuracy, and compare it with a single Decision Tree.

from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100, random_state=42
)
bagging.fit(X_train, y_train)
bag_pred = bagging.predict(X_test)

print(f"Single Decision Tree accuracy: {accuracy_score(y_test, tree_pred):.4f}")
print(f"Bagging Classifier accuracy: {accuracy_score(y_test, bag_pred):.4f}")

Single Decision Tree accuracy: 0.9333
Bagging Classifier accuracy: 0.9667


In [11]:
#Question 8: Write a Python program to
#train a Random Forest Classifier,
#tune max_depth and n_estimators using GridSearchCV, and
#print the best parameters and final accuracy.

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

grid = {
    "max_depth": [None, 5, 10, 15],
    "n_estimators": [100, 200]
}
search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=grid, cv=5, scoring="accuracy", n_jobs=-1
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
pred = best_model.predict(X_test)

print("Best parameters:", search.best_params_)
print(f"Final test accuracy: {accuracy_score(y_test, pred):.4f}")

Best parameters: {'max_depth': None, 'n_estimators': 200}
Final test accuracy: 0.9561


In [12]:
#Question 9: Write a Python program to
#train a Bagging Regressor and a Random Forest Regressor on the California Housing dataset and
#compare their Mean Squared Errors (MSE).

from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# The first run may download the California Housing dataset.
data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=100, random_state=42, n_jobs=-1
)
forest = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)
bag.fit(X_train, y_train)
forest.fit(X_train, y_train)

bag_pred = bag.predict(X_test)
forest_pred = forest.predict(X_test)
print(f"Bagging Regressor MSE: {mean_squared_error(y_test, bag_pred):.4f}")
print(f"Random Forest Regressor MSE: {mean_squared_error(y_test, forest_pred):.4f}")

Bagging Regressor MSE: 0.2559
Random Forest Regressor MSE: 0.2554


Question 10: You are working as a data scientist at a financial institution to predict loan default. You have access to customer demographic and transaction history data.
Explain your step-by-step approach to
choose between Bagging or Boosting,
handle overfitting,
select base models,
evaluate performance using cross-validation, and
justify how ensemble learning improves decision-making in this real-world context.

Ans -
Step 1 — Define the prediction target and prevent data leakage.
Define default clearly (for example, default within a specified period after loan origination). Use only information available at the time the prediction would be made. Remove direct identifiers and any variables recorded after default. Split data by time where appropriate so the test reflects future applicants. Review privacy, fairness, and applicable lending requirements.

Step 2 — Prepare the data.
Inspect missing values, duplicates, outliers, class imbalance, and inconsistent categories. Impute missing values and encode categorical variables inside a scikit-learn Pipeline so preprocessing is learned only from training folds. Consider class weights or resampling within training folds if defaults are rare; never resample the validation or test set.

Step 3 — Choose Bagging or Boosting.
Start with a simple baseline such as logistic regression, then compare tree ensembles. Choose Bagging/Random Forest when a robust nonlinear model with limited tuning and reduced variance is desirable. Consider Boosting (such as HistGradientBoosting or another approved gradient-boosting implementation) when predictive performance is the priority and careful tuning is feasible. Decide using cross-validated metrics, calibration, stability, operational needs, and explainability—not accuracy alone.

Step 4 — Select base models.
For Bagging, use decision trees as base learners because they capture nonlinear relationships and interactions; bootstrap aggregation reduces the instability of individual trees. Random Forest adds random feature selection. For Boosting, use shallow decision trees or other weak learners added sequentially. Compare against logistic regression and a single decision tree as interpretable baselines.

Step 5 — Control overfitting.
Use a separate test set, cross-validation, and hyperparameter search. For Random Forest, tune `max_depth`, `min_samples_leaf`, `max_features`, and `n_estimators`. For Boosting, tune learning rate, number of iterations/trees, tree depth, and regularization; use early stopping where supported. Keep pipelines inside cross-validation, limit unnecessary features, and avoid repeated tuning against the test set.

Step 6 — Evaluate with cross-validation and relevant metrics.
Use stratified k-fold cross-validation for independent observations, or time-based validation when the data are temporal. Compare ROC-AUC and PR-AUC, precision, recall, F1, confusion matrix, and calibration. Because false negatives (missed defaults) and false positives (incorrectly flagged reliable borrowers) have different costs, choose the operating threshold using business costs and policy constraints. Report mean and variability across folds, then evaluate once on an untouched test set.

Step 7 — Explain how the ensemble helps decision-making.
A well-validated ensemble can capture nonlinear relationships and interactions among demographics and transaction patterns. Aggregating trees can improve stability and ranking of risk, helping the institution prioritize manual review, set risk-based controls, and monitor portfolios. Use calibrated probabilities and a documented threshold rather than treating a score as a final decision by itself. Monitor drift and performance after deployment, check fairness across relevant groups, provide appropriate explanations, and keep human oversight for consequential lending decisions.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score

# df must contain the target column 'default' and predictor columns.
X = df.drop(columns=["default"])
y = df["default"]

numeric = X.select_dtypes(include="number").columns
categorical = X.select_dtypes(exclude="number").columns
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

model = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=10,
        class_weight="balanced", random_state=42
    ))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    model, X, y, cv=cv,
    scoring={"roc_auc": "roc_auc", "precision": "precision",
             "recall": "recall", "f1": "f1"},
    n_jobs=-1
)
for metric in ["test_roc_auc", "test_precision", "test_recall", "test_f1"]:
    print(f"{metric}: {scores[metric].mean():.3f} ± {scores[metric].std():.3f}")